## 1. 파일 경로 설정 및 라이브러리 임포트

In [6]:
"""
===================================================================================
1. 파일 경로 설정 및 라이브러리 임포트
===================================================================================
목적: 공원온도 계산에 필요한 라이브러리와 데이터 파일 경로를 설정합니다.

주요 라이브러리:
- pandas: 데이터프레임 처리
- numpy: 수치 연산
- json: JSON 파일 읽기
- requests: 카카오 지오코딩 API 호출
- glob: 여러 CSV 파일 찾기 (24개 자치구 공원 데이터)
===================================================================================
"""

import pandas as pd
import numpy as np
import json
import requests
import time
from math import radians, sin, cos, asin, sqrt
from pathlib import Path
import glob

# 파일 경로 설정
# BASE_DIR: 프로젝트 루트 디렉토리
BASE_DIR = Path(r"c:\Users\Playdata\OneDrive\바탕 화면\ondo")

# PARK_DIR: 도시공원 데이터 폴더 (자치구별 CSV 파일)
PARK_DIR = BASE_DIR / "ondo_park" / "data" / "도시공원"

# LAND_DIR: 부동산 매물 데이터 폴더
LAND_DIR = BASE_DIR / "land_data" / "land"

# 부동산 매물 JSON 파일
LISTING_JSON = LAND_DIR / "00_통합_원투룸_2.json"

# 시작 메시지
print("=" * 60)
print("공원온도 (Park Temperature) 계산")
print("=" * 60)
print(f"\n데이터 폴더:")
print(f"  공원 데이터: {PARK_DIR}")
print(f"  부동산 데이터: {LAND_DIR}")
print(f"\n매물 파일: {LISTING_JSON.name}")

공원온도 (Park Temperature) 계산

데이터 폴더:
  공원 데이터: c:\Users\Playdata\OneDrive\바탕 화면\ondo\ondo_park\data\도시공원
  부동산 데이터: c:\Users\Playdata\OneDrive\바탕 화면\ondo\land_data\land

매물 파일: 00_통합_원투룸_2.json


## 2. 부동산 매물 데이터 로드

In [ ]:
"""
===================================================================================
2. 부동산 매물 데이터 로드
===================================================================================
목적:
1. JSON 형식의 부동산 매물 데이터 로드
2. 매물번호와 주소 정보 추출
3. 지오코딩 준비 (다음 단계에서 주소 → 위도/경도 변환)

데이터 구조:
- 매물번호: 고유 식별자
- 주소: 전체주소 (카카오 API로 지오코딩할 텍스트)
===================================================================================
"""

print("\n" + "=" * 60)
print("2. 부동산 매물 데이터 로드")
print("=" * 60)

# JSON 파일 로드
with open(LISTING_JSON, encoding="utf-8") as f:
    listings_data = json.load(f)

print(f"총 매물 수: {len(listings_data)}개")

# 매물 데이터프레임 생성
all_listings = []

for item in listings_data:
    # 매물번호 추출
    listing_id = item.get("매물번호", "")
    
    # 주소 정보 추출
    address_info = item.get("주소_정보", {})
    address = address_info.get("전체주소", "")
    
    # 매물번호와 주소가 모두 있는 경우만 포함
    if listing_id and address:
        all_listings.append({
            "listing_id": listing_id,
            "address": address,
            "lat": None,  # 지오코딩 후 채워질 예정
            "lng": None,  # 지오코딩 후 채워질 예정
        })

# 리스트를 DataFrame으로 변환
listings = pd.DataFrame(all_listings)

print(f"매물 데이터프레임 생성: {len(listings)}개")
print(listings.head())


2. 부동산 매물 데이터 로드
총 매물 수: 6개
매물 데이터프레임 생성: 6개
  listing_id                        address   lat   lng
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워  None  None
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운  None  None
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈  None  None
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌  None  None
4   18306808            서울특별시 서초구 양재동 366-9  None  None


## 3. 지오코딩 사용해서 위/경도 변환

In [ ]:
"""
===================================================================================
3. 지오코딩 (주소 → 위도/경도 변환)
===================================================================================
목적:
1. 카카오 API를 사용해서 텍스트 주소를 위도/경도로 변환
2. API 호출 제한을 고려한 속도 조절 (0.1초 대기)
3. 지오코딩 결과를 CSV 파일로 저장

주의사항:
- 카카오 API 무료 플랜: 일일 호출 제한 존재
- 기존에 저장된 파일이 있으면 재사용 가능
===================================================================================
"""

print("\n" + "=" * 60)
print("3. 지오코딩")
print("=" * 60)

# 카카오 지오코딩 함수 정의
def geocode_address(address, kakao_api_key="None"):
    """
    카카오 REST API를 사용해서 주소를 위도/경도로 변환
    
    Args:
        address: 검색할 주소 (예: "서울특별시 서초구 양재동 116-3")
        kakao_api_key: 카카오 개발자 앱의 REST API 키
    
    Returns:
        (위도, 경도) 튜플, 또는 실패 시 (None, None)
    """
    # API 키가 없으면 지오코딩 불가
    if not kakao_api_key:
        return None, None
    
    # 카카오 로컬 API 엔드포인트
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    
    # API 인증 헤더 (REST API 키 사용)
    headers = {"Authorization": f"KakaoAK {kakao_api_key}"}
    
    # 검색 파라미터
    params = {"query": address}
    
    try:
        # GET 요청 전송
        response = requests.get(url, headers=headers, params=params)
        
        # 성공 응답 확인 (HTTP 200)
        if response.status_code == 200:
            result = response.json()
            
            # 검색 결과가 있으면 첫 번째 결과 사용
            if result['documents']:
                doc = result['documents'][0]
                # y = 위도(latitude), x = 경도(longitude)
                return float(doc['y']), float(doc['x'])
    except:
        pass
    
    return None, None

# 카카오 API 키 설정
KAKAO_API_KEY = "2b552ce8cd4023635129f04f72cfd7f7"

if KAKAO_API_KEY:
    print("카카오 API를 사용해서 지오코딩을 수행합니다...")
    print(f"총 {len(listings)}개 주소 처리 예정")
    
    # 각 매물의 주소를 지오코딩
    for idx, row in listings.iterrows():
        # 진행 상황 출력 (100개마다)
        if idx % 100 == 0:
            print(f"  진행: {idx}/{len(listings)}")
        
        # 카카오 API 호출하여 위도/경도 받기
        lat, lng = geocode_address(row['address'], KAKAO_API_KEY)
        
        # DataFrame에 저장
        listings.at[idx, 'lat'] = lat
        listings.at[idx, 'lng'] = lng
        
        # API 호출 제한 방지: 0.1초 대기
        time.sleep(0.1)
    
    # 지오코딩 성공률 확인
    success_count = listings[['lat', 'lng']].notna().all(axis=1).sum()
    print(f"\n지오코딩 성공: {success_count}/{len(listings)} ({success_count/len(listings)*100:.1f}%)")
    
    # 지오코딩 결과를 CSV 파일로 저장
    listings.to_csv(BASE_DIR / "park_listings_with_coords.csv", index=False, encoding="utf-8-sig")
    print("지오코딩 결과를 'park_listings_with_coords.csv'에 저장했습니다.")
    
else:
    print("카카오 API 키가 설정되지 않았습니다.")
    print("\n기존 파일이 있다면 로드하세요:")
    
    # 기존 파일 확인
    saved_file = BASE_DIR / "park_listings_with_coords.csv"
    if saved_file.exists():
        print(f"\n기존 파일을 발견했습니다: {saved_file}")
        listings = pd.read_csv(saved_file, encoding="utf-8-sig")
        print(f"로드 완료: {len(listings)}개 매물")
    else:
        print("\n경고: 기존 파일이 없습니다.")
        print("공원온도 계산을 위해서는 위도/경도가 필요합니다.")

# 유효한 좌표가 있는 매물만 필터링
# lat과 lng가 모두 NaN이 아닌 행만 선택
listings_valid = listings[listings[['lat', 'lng']].notna().all(axis=1)].copy()
print(f"\n유효한 좌표가 있는 매물: {len(listings_valid)}개")

# 결과 확인
if len(listings_valid) > 0:
    print(listings_valid.head())


3. 지오코딩
카카오 API를 사용해서 지오코딩을 수행합니다...
총 6개 주소 처리 예정
  진행: 0/6

지오코딩 성공: 6/6 (100.0%)
지오코딩 결과를 'park_listings_with_coords.csv'에 저장했습니다.

유효한 좌표가 있는 매물: 6개
  listing_id                        address        lat         lng
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워  37.474678  127.035655
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운  37.465608  127.034504
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈  37.474999  127.041886
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌   37.46933  127.044146
4   18306808            서울특별시 서초구 양재동 366-9  37.468881  127.045828


## 4. 도시공원 데이터 로드

In [ ]:
"""
===================================================================================
4. 서울시 전체 도시공원 데이터 로드
===================================================================================
목적:
1. 서울시 24개 자치구의 도시공원 데이터를 통합
2. 각 공원의 위치(위도/경도), 면적, 시설 정보 수집

데이터 구조:
- 파일 형식: 자치구별 CSV 파일 (예: "공원_서울특별시강남구.csv")
- 총 1,715개 도시공원 (어린이공원, 근린공원, 소공원 등)
===================================================================================
"""

print("\n" + "=" * 60)
print("1. 도시공원 데이터 로드")
print("=" * 60)

# 도시공원 폴더에서 모든 CSV 파일 찾기
# glob("*.csv"): 확장자가 .csv인 모든 파일 검색
park_files = list(PARK_DIR.glob("*.csv"))
print(f"\n발견된 공원 데이터 파일: {len(park_files)}개")

# 모든 공원 데이터 통합
parks_list = []

for park_file in park_files:
    try:
        # CSV 파일 로드
        df = pd.read_csv(park_file, encoding="utf-8")
        
        # 파일명에서 자치구 이름 추출
        # 예: "공원_서울특별시강남구.csv" → ["공원", "서울특별시강남구"] → "강남구"
        district = park_file.stem.split("_")[1].replace("서울특별시", "")
        df["자치구"] = district
        
        # 리스트에 추가
        parks_list.append(df)
        print(f"  ✓ {district}: {len(df)}개 공원")
        
    except Exception as e:
        print(f"  ✗ {park_file.name}: 오류 - {e}")

# 모든 자치구 데이터를 하나의 DataFrame으로 통합
# ignore_index=True: 인덱스를 0부터 다시 시작
parks = pd.concat(parks_list, ignore_index=True)

print(f"\n총 공원 수: {len(parks)}개")
print(f"컬럼: {parks.columns.tolist()}")

# 샘플 데이터 확인
print("\n샘플 데이터:")
print(parks.head())


1. 도시공원 데이터 로드

발견된 공원 데이터 파일: 24개
  ✓ 강남구: 133개 공원
  ✓ 강동구: 74개 공원
  ✓ 강북구: 54개 공원
  ✓ 강서구: 151개 공원
  ✓ 관악구: 75개 공원
  ✓ 광진구: 43개 공원
  ✓ 구로구: 62개 공원
  ✓ 금천구: 52개 공원
  ✓ 노원구: 4개 공원
  ✓ 동대문구: 50개 공원
  ✓ 동작구: 52개 공원
  ✓ 마포구: 90개 공원
  ✓ 서대문구: 67개 공원
  ✓ 서초구: 128개 공원
  ✓ 성동구: 48개 공원
  ✓ 성북구: 53개 공원
  ✓ 송파구: 140개 공원
  ✓ 양천구: 96개 공원
  ✓ 영등포구: 50개 공원
  ✓ 용산구: 49개 공원
  ✓ 은평구: 105개 공원
  ✓ 종로구: 43개 공원
  ✓ 중구: 38개 공원
  ✓ 중랑구: 58개 공원

총 공원 수: 1715개
컬럼: ['관리번호', '공원명', '공원구분', '소재지도로명주소', '소재지지번주소', '위도', '경도', '공원면적', '공원보유시설(운동시설)', '공원보유시설(유희시설)', '공원보유시설(편익시설)', '공원보유시설(교양시설)', '공원보유시설(기타시설)', '지정고시일', '관리기관명', '전화번호', '데이터기준일자', '자치구']

샘플 데이터:
          관리번호      공원명  공원구분 소재지도로명주소              소재지지번주소         위도  \
0  11680-00002   신사근린공원  근린공원      NaN   서울특별시 강남구 압구정동 422  37.526708   
1  11680-00003   대청근린공원  근린공원      NaN    서울특별시 강남구 일원동 621  37.492416   
2  11680-00004  늘푸른근린공원  근린공원      NaN    서울특별시 강남구 일원동 690  37.489511   
3  11680-00005  개포서근린공원  근린공원      NaN    서울특별시 강남구 개포동 180

## 5. 공원 점수 계산

In [10]:
"""
===================================================================================
5. 공원 점수 계산 (각 공원의 품질 점수)
===================================================================================
목적:
1. 각 공원의 품질을 3가지 기준으로 평가
2. 크기(40%) + 시설(40%) + 유형(20%) 가중치로 종합 점수 산출

점수 구성:
- 크기 점수: 공원면적이 클수록 높은 점수
- 시설 점수: 운동/유희/편익/교양/기타 시설이 많을수록 높은 점수
- 유형 점수: 근린공원 > 어린이공원 > 소공원 순으로 점수 부여
===================================================================================
"""

print("\n" + "=" * 60)
print("4. 공원 점수 계산")
print("=" * 60)

# Min-Max 정규화 함수 정의
def minmax_norm(s: pd.Series) -> pd.Series:
    """
    시리즈를 0~1 범위로 정규화
    
    Args:
        s: pandas Series (정규화할 데이터)
    
    Returns:
        정규화된 Series (0~1 사이 값)
    """
    s_min, s_max = s.min(), s.max()
    
    # 최대값과 최소값이 같으면 0 반환 (분산이 없는 경우)
    if s_max - s_min == 0:
        return pd.Series(0.0, index=s.index)
    
    # Min-Max 정규화 적용
    return (s - s_min) / (s_max - s_min)

# === 1) 크기 점수 (40% 가중치) ===
# 공원면적을 숫자로 변환 (오류 시 0)
parks["공원면적"] = pd.to_numeric(parks["공원면적"], errors='coerce').fillna(0)

# Min-Max 정규화: 가장 작은 공원 = 0.0, 가장 큰 공원 = 1.0
parks["size_score"] = minmax_norm(parks["공원면적"])

print("1) 크기 점수 계산 완료")
print(f"   공원면적 범위: {parks['공원면적'].min():.1f} ~ {parks['공원면적'].max():.1f} m²")

# === 2) 시설 풍부도 점수 (40% 가중치) ===
# 5가지 시설 유형 컬럼
facility_cols = [
    "공원보유시설(운동시설)",    # 예: 농구장, 운동기구 등
    "공원보유시설(유희시설)",    # 예: 그네, 시소, 조합놀이대 등
    "공원보유시설(편익시설)",    # 예: 화장실, 음수대, 파고라 등
    "공원보유시설(교양시설)",    # 예: 도서관, 전시관 등
    "공원보유시설(기타시설)",    # 예: 관리사무소 등
]

# 각 시설 컬럼이 비어있지 않으면 1, 비어있으면 0
for col in facility_cols:
    if col in parks.columns:
        # notna(): NaN이 아닌 값은 True
        # astype(int): True→1, False→0으로 변환
        parks[col + "_flag"] = parks[col].notna().astype(int)
    else:
        parks[col + "_flag"] = 0

# 시설 개수 합계 (0~5개)
parks["facility_count"] = parks[[c + "_flag" for c in facility_cols]].sum(axis=1)

# Min-Max 정규화
parks["facility_score"] = minmax_norm(parks["facility_count"])

print("2) 시설 풍부도 점수 계산 완료")
print(f"   시설 개수 범위: {parks['facility_count'].min():.0f} ~ {parks['facility_count'].max():.0f}개")

# === 3) 공원 유형 점수 (20% 가중치) ===
# 공원 유형별 고정 점수
type_map = {
    "근린공원": 1.0,      # 대규모 공원, 다양한 시설
    "어린이공원": 0.8,    # 어린이 전용 놀이터
    "소공원": 0.7,        # 작은 공원
    # 기타 공원은 0.9점 기본값
}

# map(): 딕셔너리 값으로 변환, 없으면 0.9
parks["type_score"] = parks["공원구분"].map(type_map).fillna(0.9)

print("3) 공원구분 점수 계산 완료")
print(f"   공원 유형: {parks['공원구분'].value_counts().to_dict()}")

# === 4) 공원 최종 점수 (S_park) ===
# 가중 평균: 크기 40% + 시설 40% + 유형 20%
parks["S_park"] = (
    0.4 * parks["size_score"]       # 크기 기여도
    + 0.4 * parks["facility_score"] # 시설 기여도
    + 0.2 * parks["type_score"]     # 유형 기여도
)

print("4) 공원 최종 점수 계산 완료")
print(f"   S_park 범위: {parks['S_park'].min():.3f} ~ {parks['S_park'].max():.3f}")

# 공원 점수 샘플 출력
print("\n공원 점수 샘플:")
print(parks[["공원명", "자치구", "공원구분", "공원면적", "facility_count", "S_park"]].head(10))


4. 공원 점수 계산
1) 크기 점수 계산 완료
   공원면적 범위: 48.8 ~ 6691885.3 m²
2) 시설 풍부도 점수 계산 완료
   시설 개수 범위: 0 ~ 5개
3) 공원구분 점수 계산 완료
   공원 유형: {'어린이공원': 1012, '근린공원': 332, '소공원': 263, '문화공원': 31, '기타공원': 29, '역사공원': 17, '기타': 15, '수변공원': 6, '체육공원': 4, '주제공원': 2, '묘지공원': 1, '기타(가로공원)': 1, '마을마당': 1, '도시농업공원': 1}
4) 공원 최종 점수 계산 완료
   S_park 범위: 0.140 ~ 0.661

공원 점수 샘플:
       공원명  자치구  공원구분     공원면적  facility_count    S_park
0   신사근린공원  강남구  근린공원   6800.0               2  0.360404
1   대청근린공원  강남구  근린공원  14089.1               3  0.440839
2  늘푸른근린공원  강남구  근린공원  11410.3               3  0.440679
3  개포서근린공원  강남구  근린공원  11219.2               2  0.360668
4  개포동근린공원  강남구  근린공원   9705.6               2  0.360577
5   대치근린공원  강남구  근린공원  10408.9               3  0.440619
6   늘벗근린공원  강남구  근린공원  19812.3               3  0.441181
7   독골근린공원  강남구  근린공원  10022.9               3  0.440596
8   포이근린공원  강남구  근린공원  16811.4               2  0.361002
9   청룡근린공원  강남구  근린공원   9367.0               2  0.360557


## 6.  거리 계산 함수

In [11]:
"""
===================================================================================
6. 거리 계산 함수
===================================================================================
목적:
1. 위도/경도로 두 지점 간 실제 거리 계산 (Haversine 공식)
2. 거리에 따른 점수 함수 정의

거리 점수 설계:
- 도보 10분(약 800m) 이내를 의미 있는 범위로 설정
- 거리가 멀어질수록 점수가 지수적으로 감소
===================================================================================
"""

print("\n" + "=" * 60)
print("5. 거리 계산 함수")
print("=" * 60)

def haversine(lat1, lon1, lat2, lon2):
    """
    Haversine 공식을 사용한 두 지점 간 거리 계산
    
    Args:
        lat1, lon1: 첫 번째 지점의 위도, 경도 (도 단위)
        lat2, lon2: 두 번째 지점의 위도, 경도 (도 단위)
    
    Returns:
        거리 (미터 단위)
    
    공식 설명:
    1. 위도/경도를 라디안으로 변환
    2. Haversine 공식으로 중심각 계산
    3. 중심각 × 지구 반지름 = 거리
    """
    R = 6371000  # 지구 반지름 (미터)
    
    # 도(degree)를 라디안(radian)으로 변환
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # 위도/경도 차이
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Haversine 공식
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))  # 중심각
    
    return R * c  # 거리 = 반지름 × 중심각

def distance_score(d):
    """
    공원까지 거리에 따른 점수 계산
    
    Args:
        d: 공원까지의 거리 (미터)
    
    Returns:
        거리 점수 (0~1 범위)
    
    점수 공식:
        distance_score(d) = exp(-d / 400)
    
    거리별 점수:
    - d = 0m   → 1.000 (바로 앞)
    - d = 200m → 0.607 (도보 2-3분)
    - d = 400m → 0.368 (도보 5분)
    - d = 600m → 0.223 (도보 7-8분)
    - d = 800m → 0.135 (도보 10분)
    
    설계 의도:
    - 400m를 기준점으로 설정 (도보 5분)
    - 거리가 400m 증가할 때마다 점수가 e배 감소
    - 800m(도보 10분)를 넘으면 점수가 급격히 낮아짐
    """
    return float(np.exp(-d / 400.0))

print("거리 계산 함수 정의 완료")

# 거리별 점수 예시 출력
print("\n거리별 점수 예시:")
for dist in [0, 200, 400, 600, 800]:
    print(f"  {dist}m: {distance_score(dist):.3f}")


5. 거리 계산 함수
거리 계산 함수 정의 완료

거리별 점수 예시:
  0m: 1.000
  200m: 0.607
  400m: 0.368
  600m: 0.223
  800m: 0.135


## 7.  매물별로 공원 점수 계산

In [12]:
"""
===================================================================================
7. 매물별 공원 점수 계산
===================================================================================
목적:
1. 각 매물 주변의 공원들을 찾아 점수 계산
2. 반경 800m 이내 공원들의 거리 가중 평균 산출

계산 방법:
1. 매물에서 모든 공원까지의 거리 계산
2. 반경 800m 이내 공원들만 선택 (없으면 가장 가까운 공원 1개)
3. 각 공원의 점수 = 공원 품질(S_park) × 거리 점수
4. 거리 가중치로 평균 계산 (가까운 공원일수록 높은 가중치)
===================================================================================
"""

print("\n" + "=" * 60)
print("6. 매물별 공원 점수 계산")
print("=" * 60)

# 공원 좌표 및 점수를 numpy 배열로 준비 (속도 최적화)
# 위도/경도가 유효한 공원만 사용
parks_valid = parks[parks[["위도", "경도"]].notna().all(axis=1)].copy()
print(f"유효한 좌표가 있는 공원: {len(parks_valid)}개 / {len(parks)}개")

# numpy 배열로 변환 (벡터화 연산으로 속도 향상)
park_coords = parks_valid[["위도", "경도"]].to_numpy()
park_scores = parks_valid["S_park"].to_numpy()

def compute_park_score_for_property(p_lat, p_lng, radius=800):
    """
    매물 1건에 대한 공원 점수 계산
    
    Args:
        p_lat, p_lng: 매물의 위도, 경도
        radius: 고려할 공원의 최대 거리 (미터, 기본값 800m = 도보 10분)
    
    Returns:
        매물의 공원 점수 (0~1 범위)
    
    알고리즘:
    1. 매물에서 모든 공원까지의 거리 계산
    2. radius 이내 공원 필터링
       - 만약 반경 내 공원이 없으면 가장 가까운 공원 1개 사용
    3. 각 공원에 대해:
       - 거리 점수 계산: exp(-d/400)
       - 거리 가중치 계산: w = 1/(1 + d/300)
       - 종합 점수 = 공원점수 × 거리점수
    4. 가중 평균 반환
    
    예시:
    - 공원 A: 200m, S_park=0.5
      → 거리점수=0.607, 가중치=0.6, 종합=0.304
    - 공원 B: 500m, S_park=0.8
      → 거리점수=0.287, 가중치=0.375, 종합=0.230
    - 최종 점수 = (0.304×0.6 + 0.230×0.375) / (0.6+0.375) = 0.276
    """
    # 공원 데이터가 없으면 0점
    if len(park_coords) == 0:
        return 0.0
    
    # 매물에서 모든 공원까지의 거리 계산
    dists = np.array([
        haversine(p_lat, p_lng, park_lat, park_lng)
        for (park_lat, park_lng) in park_coords
    ])
    
    # radius 이내 공원 필터링
    mask = dists <= radius
    
    if not mask.any():
        # 반경 내 공원이 없으면 가장 가까운 공원 1개라도 사용
        idx_min = dists.argmin()
        dists_use = dists[[idx_min]]
        scores_use = park_scores[[idx_min]]
    else:
        # 반경 내 공원들 사용
        dists_use = dists[mask]
        scores_use = park_scores[mask]
    
    # 거리 점수 계산 (각 공원까지의 거리를 점수로 변환)
    d_score = np.array([distance_score(d) for d in dists_use])
    
    # 거리 가중치 계산 (가까울수록 높은 가중치)
    # w = 1/(1 + d/300)
    # 예: d=0m → w=1.0, d=300m → w=0.5, d=600m → w=0.33
    w = 1 / (1 + dists_use / 300.0)
    
    # 공원 점수와 거리 점수를 결합
    # combined = S_park × distance_score
    combined = scores_use * d_score
    
    # 거리 가중 평균 반환
    # Σ(combined × w) / Σ(w)
    return float(np.sum(combined * w) / np.sum(w))

# 매물별 공원 점수 계산
if len(listings_valid) > 0:
    print("매물별 공원 점수 계산 중...")
    park_scores_property = []
    
    for idx, row in listings_valid.iterrows():
        # 진행 상황 출력 (500개마다)
        if idx % 500 == 0:
            print(f"  진행: {idx}/{len(listings_valid)}")
        
        # 공원 점수 계산
        score = compute_park_score_for_property(row["lat"], row["lng"])
        park_scores_property.append(score)
    
    # DataFrame에 저장
    listings_valid["S_park_property_raw"] = park_scores_property
    
    print("\n공원 점수 계산 완료!")
    print(listings_valid["S_park_property_raw"].describe())
    # count: 매물 개수
    # mean: 평균 공원 점수
    # std: 표준편차
    # min~max: 최소~최대 공원 점수
    
else:
    print("유효한 매물이 없습니다. 지오코딩을 먼저 수행하세요.")


6. 매물별 공원 점수 계산
유효한 좌표가 있는 공원: 1715개 / 1715개
매물별 공원 점수 계산 중...
  진행: 0/6

공원 점수 계산 완료!
count    6.000000
mean     0.072046
std      0.006392
min      0.065301
25%      0.066833
50%      0.071622
75%      0.075685
max      0.081456
Name: S_park_property_raw, dtype: float64


In [13]:
"""
===================================================================================
8. 최종 공원온도 계산
===================================================================================
목적:
1. 공원 점수를 정규화하여 0~1 범위로 조정
2. 30~43°C 스케일의 "공원온도"로 변환
3. 결과를 CSV로 저장하고 분석

공원온도 공식:
- S_park_norm = (S_park_raw - min) / (max - min)  # 0~1 정규화
- ParkTemp = 30 + 13 × S_park_norm  # 온도 변환

온도 해석:
- 30-34°C: 공원 환경이 부족 (공원 적음/멀음/작음)
- 34-39°C: 평균~양호 (일반적인 서울 주거지)
- 39-43°C: 공원 환경 우수 (가까운 공원 많음/크고 시설 좋음)
===================================================================================
"""

print("\n" + "=" * 60)
print("7. 최종 공원온도 계산")
print("=" * 60)

if len(listings_valid) > 0:
    # Min-Max 정규화 함수 (컬럼 단위)
    def minmax_norm_column(df, col):
        """
        DataFrame의 특정 컬럼을 0~1 범위로 정규화
        
        Args:
            df: DataFrame
            col: 정규화할 컬럼명
        
        Returns:
            정규화된 Series
        """
        s = df[col]
        s_min, s_max = s.min(), s.max()
        
        # 최대값과 최소값이 같으면 0 반환
        if s_max - s_min == 0:
            return pd.Series(0.0, index=df.index)
        
        # Min-Max 정규화 적용
        return (s - s_min) / (s_max - s_min)
    
    # 공원 점수 정규화 (0~1 범위)
    # 각 매물의 상대적 순위를 0~1로 변환
    listings_valid["S_park_norm"] = minmax_norm_column(listings_valid, "S_park_property_raw")
    
    # 공원온도 계산 (30~43°C 스케일)
    # ParkTemp = 30 + 13 × S_park_norm
    #
    # 공식 설명:
    # - S_park_norm = 0.0 → 30°C (최저 온도, 공원 환경 불량)
    # - S_park_norm = 0.5 → 36.5°C (평균 온도, 보통 수준)
    # - S_park_norm = 1.0 → 43°C (최고 온도, 공원 환경 우수)
    #
    # 대안 공식: ParkTemp = 36.5 + 13 × (S_park_norm - 0.5)
    # → 36.5°C를 중심으로 ±6.5°C 범위
    listings_valid["ParkTemp"] = 30 + 13 * listings_valid["S_park_norm"]
    
    print("공원온도 계산 완료!")
    print("\n공원온도 통계:")
    print(listings_valid["ParkTemp"].describe())
    # count: 매물 개수
    # mean: 평균 온도
    # std: 표준편차
    # min: 최저 온도 (공원이 가장 부족한 매물)
    # max: 최고 온도 (공원이 가장 좋은 매물)
    
    # 결과 확인 (샘플)
    print("\n샘플 결과:")
    result_cols = ["listing_id", "address", "S_park_property_raw", "S_park_norm", "ParkTemp"]
    print(listings_valid[result_cols].head(10))
    
    # 결과 저장
    output_file = BASE_DIR / "listings_with_park_temp.csv"
    listings_valid.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n결과를 '{output_file}'에 저장했습니다.")
    
    # 공원온도 분포 확인
    print("\n공원온도 구간별 분포:")
    bins = [30, 34, 39, 43]  # 구간 경계
    labels = ["낮음 (30-34°C)", "보통 (34-39°C)", "높음 (39-43°C)"]
    
    # 구간별 분류
    listings_valid["ParkTemp_category"] = pd.cut(
        listings_valid["ParkTemp"], 
        bins=bins,           # 구간 경계값
        labels=labels,       # 구간 레이블
        include_lowest=True  # 최소값 포함
    )
    
    # 구간별 매물 수 출력
    print(listings_valid["ParkTemp_category"].value_counts().sort_index())
    
    # 해석:
    # - 낮음: 공원이 부족하거나 멀거나 작음
    # - 보통: 일반적인 서울 주거지 수준의 공원 환경
    # - 높음: 가까운 거리에 크고 시설이 좋은 공원이 많음
    
else:
    print("유효한 매물이 없습니다. 지오코딩을 먼저 수행하세요.")


7. 최종 공원온도 계산
공원온도 계산 완료!

공원온도 통계:
count     6.000000
mean     35.428058
std       5.143452
min      30.000000
25%      31.233012
50%      35.086912
75%      38.356336
max      43.000000
Name: ParkTemp, dtype: float64

샘플 결과:
  listing_id                        address  S_park_property_raw  S_park_norm  \
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워             0.076975     0.722656   
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운             0.071430     0.379388   
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈             0.065301     0.000000   
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌             0.081456     1.000000   
4   18306808            서울특별시 서초구 양재동 366-9             0.071815     0.403213   
5   18443369  서울특별시 서초구 양재동 257-8 다비치 스위트 홈             0.065301     0.000000   

    ParkTemp  
0  39.394523  
1  34.932049  
2  30.000000  
3  43.000000  
4  35.241774  
5  30.000000  

결과를 'c:\Users\Playdata\OneDrive\바탕 화면\ondo\listings_with_park_temp.csv'에 저장했습니다.

공원온도 구간별 분포:
Par

In [ ]:
# # 8. 추가 분석 - 자치구별 공원온도 통계
# print("\n" + "=" * 60)
# print("8. 자치구별 공원온도 통계")
# print("=" * 60)

# if len(listings_valid) > 0:
#     # 주소에서 자치구 추출
#     def extract_district(address):
#         """주소에서 자치구 추출 (예: '서울특별시 강남구 ...' -> '강남구')"""
#         parts = address.split()
#         for part in parts:
#             if part.endswith("구"):
#                 return part
#         return "기타"
    
#     listings_valid["자치구"] = listings_valid["address"].apply(extract_district)
    
#     # 자치구별 통계
#     district_stats = listings_valid.groupby("자치구").agg({
#         "ParkTemp": ["count", "mean", "std", "min", "max"],
#         "S_park_property_raw": "mean"
#     }).round(2)
    
#     district_stats.columns = ["매물수", "평균온도", "표준편차", "최소온도", "최대온도", "공원점수"]
#     district_stats = district_stats.sort_values("평균온도", ascending=False)
    
#     print("\n자치구별 공원온도 통계:")
#     print(district_stats)
    
#     # 자치구별 분포 저장
#     district_stats.to_csv(BASE_DIR / "park_temp_by_district.csv", encoding="utf-8-sig")
#     print("\n자치구별 통계를 'park_temp_by_district.csv'에 저장했습니다.")
    
#     # 공원온도 상위/하위 매물
#     print("\n" + "=" * 60)
#     print("공원온도 상위/하위 매물")
#     print("=" * 60)
    
#     print("\n[공원온도 상위 10개 매물]")
#     top_listings = listings_valid.nlargest(10, "ParkTemp")
#     print(top_listings[["listing_id", "address", "자치구", "ParkTemp"]])
    
#     print("\n[공원온도 하위 10개 매물]")
#     bottom_listings = listings_valid.nsmallest(10, "ParkTemp")
#     print(bottom_listings[["listing_id", "address", "자치구", "ParkTemp"]])
    
# else:
#     print("유효한 매물이 없습니다.")


8. 자치구별 공원온도 통계

자치구별 공원온도 통계:
     매물수   평균온도  표준편차  최소온도  최대온도  공원점수
자치구                                    
서초구    6  35.43  5.14  30.0  43.0  0.07

자치구별 통계를 'park_temp_by_district.csv'에 저장했습니다.

공원온도 상위/하위 매물

[공원온도 상위 10개 매물]
  listing_id                        address  자치구   ParkTemp
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌  서초구  43.000000
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워  서초구  39.394523
4   18306808            서울특별시 서초구 양재동 366-9  서초구  35.241774
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운  서초구  34.932049
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈  서초구  30.000000
5   18443369  서울특별시 서초구 양재동 257-8 다비치 스위트 홈  서초구  30.000000

[공원온도 하위 10개 매물]
  listing_id                        address  자치구   ParkTemp
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈  서초구  30.000000
5   18443369  서울특별시 서초구 양재동 257-8 다비치 스위트 홈  서초구  30.000000
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운  서초구  34.932049
4   18306808            서울특별시 서초구 양재동 366-9  서초구  35.241774
0   18446002      서울특별시 서초구 양재